In [1]:
import os
import json
import uuid
import shutil
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
SOURCE_DIR = Path("../datasets")  # 원본 PDF 위치
METADATA_FILE = SOURCE_DIR / "metadata.json"

# 이미 메타데이터 CSV가 있다면 로드 (이전 단계의 결과물 활용)
PREV_CSV = Path("../metadata_fixed.csv") 

# ---------------------------------------------------------
# 로직
# ---------------------------------------------------------

metadata_store = {}
existing_meta_df = pd.read_csv(PREV_CSV) if PREV_CSV.exists() else pd.DataFrame()

print("🚀 사이드카 패턴으로 데이터셋 마이그레이션 시작...\n")

pdf_files = list(SOURCE_DIR.glob("*.pdf"))

for pdf_path in pdf_files:
    # 이미 UUID 포맷인 파일은 건너뛰기 (중복 실행 방지)
    try:
        uuid.UUID(pdf_path.stem)
        continue
    except ValueError:
        pass # UUID가 아니면 처리 대상

    # 1. 고유 ID 생성 (UUID4)
    file_id = str(uuid.uuid4())
    new_filename = f"{file_id}.pdf"
    new_path = SOURCE_DIR / new_filename

    # 2. 메타데이터 확보
    # 이전에 CSV로 만들어둔 정보가 있으면 매칭하고, 없으면 파일명 기반으로 생성
    paper_info = {}
    
    # CSV에서 현재 파일명(혹은 이전 파일명)으로 정보 찾기
    if not existing_meta_df.empty:
        # 파일명으로 검색 (new_filename 컬럼 혹은 original_filename 컬럼 활용)
        match = existing_meta_df[
            (existing_meta_df['new_filename'] == pdf_path.name) | 
            (existing_meta_df['original_filename'] == pdf_path.name)
        ]
        
        if not match.empty:
            row = match.iloc[0]
            paper_info = {
                "title": row.get('title', pdf_path.stem),
                "year": str(row.get('year', 'Unknown')),
                "source": row.get('source', 'Manual'),
                "original_filename": row.get('original_filename', pdf_path.name)
            }
        else:
            # CSV에 없으면 현재 파일명에서 추론
            paper_info = {
                "title": pdf_path.stem,
                "year": "Unknown",
                "source": "FileSystem",
                "original_filename": pdf_path.name
            }
    
    # 3. 메타데이터 스토어에 등록
    metadata_store[file_id] = paper_info

    # 4. 파일 이름 변경 (실제 파일 시스템 조작)
    os.rename(pdf_path, new_path)
    print(f"📦 Archived: {pdf_path.name[:20]}... -> {new_filename}")

# ---------------------------------------------------------
# 메타데이터 저장 (JSON)
# ---------------------------------------------------------
if metadata_store:
    # 기존 메타데이터가 있다면 병합
    if METADATA_FILE.exists():
        with open(METADATA_FILE, 'r', encoding='utf-8') as f:
            old_data = json.load(f)
            old_data.update(metadata_store)
            metadata_store = old_data

    with open(METADATA_FILE, 'w', encoding='utf-8') as f:
        json.dump(metadata_store, f, ensure_ascii=False, indent=4)
        
    print(f"\n✅ 마이그레이션 완료!")
    print(f"   - PDF 파일들은 모두 UUID로 변경되었습니다.")
    print(f"   - 매핑 정보는 '{METADATA_FILE}'에 저장되었습니다.")
else:
    print("\n⚠️ 변경할 파일이 없습니다.")

🚀 사이드카 패턴으로 데이터셋 마이그레이션 시작...

📦 Archived: [Unknown] [Unknown]_... -> f0e48edf-9f92-4276-91e3-a8e998411bd0.pdf
📦 Archived: [Unknown] [Unknown]_... -> f79bf71b-e46f-4e85-8abe-4d74bedc76eb.pdf
📦 Archived: [Unknown] [Unknown]_... -> 5766583f-6271-4b00-95d7-f7234a363aea.pdf
📦 Archived: [Unknown] [Unknown]_... -> 09e3af80-917a-4a5a-a212-f71b32e0066b.pdf
📦 Archived: [Unknown] [Unknown]_... -> 949f0924-913a-4a7c-91d3-b87be69ce944.pdf
📦 Archived: [Unknown] [Unknown]_... -> abe6742c-be99-4cb4-86e6-1917bac8b44b.pdf
📦 Archived: [Unknown] [Unknown]_... -> 5ecdfc5e-2b74-4fa0-9336-8d8eec08a5d6.pdf
📦 Archived: [Unknown] [Unknown]_... -> 9d7dd729-ce3b-492f-b30f-1deaab764477.pdf
📦 Archived: [Unknown] [Unknown]_... -> 37499ba8-d3c1-4fda-89ec-f2d77fc417e1.pdf
📦 Archived: [Unknown] [Unknown]_... -> b42d0916-0ef0-4a47-a1cb-791a95757830.pdf

✅ 마이그레이션 완료!
   - PDF 파일들은 모두 UUID로 변경되었습니다.
   - 매핑 정보는 '../datasets/metadata.json'에 저장되었습니다.


In [3]:
# RAG 시스템에서 데이터를 로드하는 방식
import json
from pathlib import Path

# 1. 메타데이터 로드
with open('../datasets/metadata.json', 'r') as f:
    library = json.load(f)

# 2. 특정 논문 찾기 (예: 'Attention'이 들어간 논문)
query = "Attention"
found_papers = []

for file_id, meta in library.items():
    if query.lower() in meta['title'].lower():
        found_papers.append((file_id, meta))

# 3. 결과 출력 및 파일 경로 확인
for file_id, meta in found_papers:
    file_path = Path(f"../datasets/{file_id}.pdf")
    print(f"Title: {meta['title']}")
    print(f"Path: {file_path} (Exists: {file_path.exists()})")
    print("-" * 20)

KeyError: 'title'